# Danh gia mo hinh – Multilingual MT
Sinh ra cac metric: **Val Loss, Perplexity, BLEU Score** cho ca 2 mo hinh.

**Yeu cau truoc khi chay:**
- Dat `best_transformer_model.pt` vao thu muc `model_assets/`
- Dat `best_lstm_model.pt` vao thu muc `model_assets/` (neu co)
- Chay kernel: `Python (Multilingual MT)`

## Cell 1 – Setup duong dan & thu vien

In [2]:
# Cell 0 – Cài đặt môi trường (thêm vào đầu notebook)
!git lfs install
!git clone https://github.com/Kaiser2484/Multilingual_MT.git
%cd Multilingual_MT

import sys
sys.path.insert(0, '/content/Multilingual_MT')

# Cập nhật đường dẫn
PROJECT_DIR    = '/content/Multilingual_MT'
MODEL_DIR      = f'{PROJECT_DIR}/model_assets'
TF_CKPT        = f'{MODEL_DIR}/best_transformer_model.pt'
LSTM_CKPT      = f'{MODEL_DIR}/best_baseline_model.pt'
TEST_FILE      = f'{PROJECT_DIR}/data/processed/test.txt'
TOKENIZER_PATH = f'{PROJECT_DIR}/tokenizer/tokenizer.json'

Git LFS initialized.
Cloning into 'Multilingual_MT'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 85 (delta 28), reused 69 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 653.91 KiB | 19.23 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Filtering content: 100% (4/4), 1.02 GiB | 5.67 MiB/s, done.
/content/Multilingual_MT


In [8]:
# Chạy script tiền xử lý để tạo file test.txt và các file liên quan
!python src/prepare_data.py

# Kiểm tra lại sự tồn tại của file sau khi chạy script
import os
if os.path.exists('/content/Multilingual_MT/data/processed/test.txt'):
    print("✅ Dữ liệu đã được chuẩn bị thành công tại: /content/Multilingual_MT/data/processed/test.txt")
else:
    print("❌ Vẫn không tìm thấy file. Vui lòng kiểm tra lại log của script prepare_data.py.")

[23:09:50] INFO     ==========================================
[23:09:50] INFO       Multilingual MT – Chuẩn bị dữ liệu     
[23:09:50] INFO       max_pairs=100000 | seed=42                  
[23:09:50] INFO     ==========================================
[23:09:51] INFO     NumExpr defaulting to 2 threads.
[23:09:51] INFO     TensorFlow version 2.20.0 available.
[23:09:51] INFO     JAX version 0.7.2 available.
[23:09:51] INFO     ──────────────────────────────────────────
[23:09:51] INFO     Đang tải en-ja từ HuggingFace (Helsinki-NLP/opus-100) ...
[23:09:52] INFO     HTTP Request: HEAD https://huggingface.co/datasets/Helsinki-NLP/opus-100/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
[23:09:52] WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[23:09:52] INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Helsinki-NLP/opus-100/805090dc28bf78897da9641cdf

## Cell 2 – Load model Transformer

In [9]:
import torch
import math

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from src.models.transformer import Transformer

tf_ckpt = torch.load(TF_CKPT, map_location=DEVICE)
tf_cfg  = tf_ckpt['model_config']

transformer = Transformer(**tf_cfg).to(DEVICE)
transformer.load_state_dict(tf_ckpt['model_state'])
transformer.eval()

tf_val_loss = tf_ckpt['val_loss']
tf_ppl      = math.exp(tf_val_loss)

print(f'Transformer da load:')
print(f'  Epoch     : {tf_ckpt["epoch"]}')
print(f'  Val Loss  : {tf_val_loss:.4f}')
print(f'  Perplexity: {tf_ppl:.2f}')
total = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
print(f'  Params    : {total:,}')

Transformer da load:
  Epoch     : 35
  Val Loss  : 4.1345
  Perplexity: 62.46
  Params    : 21,905,408


## Cell 3 – Load model LSTM Baseline (bo qua neu khong co file)

In [10]:
import os
from src.models.baseline_lstm import LSTMBaseline

lstm_model    = None
lstm_val_loss = 4.25     # Gia tri da biet tu log train
lstm_ppl      = math.exp(lstm_val_loss)
lstm_bleu     = None     # Se tinh o cell sau neu co file

if os.path.exists(LSTM_CKPT):
    lstm_ckpt = torch.load(LSTM_CKPT, map_location=DEVICE)
    lstm_cfg  = lstm_ckpt['model_config']
    lstm_model = LSTMBaseline(**lstm_cfg).to(DEVICE)
    lstm_model.load_state_dict(lstm_ckpt['model_state'])
    lstm_model.eval()
    lstm_val_loss = lstm_ckpt['val_loss']
    lstm_ppl      = math.exp(lstm_val_loss)
    print(f'LSTM Baseline da load:')
    print(f'  Epoch     : {lstm_ckpt["epoch"]}')
    print(f'  Val Loss  : {lstm_val_loss:.4f}')
    print(f'  Perplexity: {lstm_ppl:.2f}')
else:
    print('Khong tim thay best_lstm_model.pt')
    print(f'Su dung gia tri da biet: Val Loss={lstm_val_loss}, PPL={lstm_ppl:.2f}')

LSTM Baseline da load:
  Epoch     : 10
  Val Loss  : 4.2456
  Perplexity: 69.80


## Cell 4 – Ham dich (Greedy Decode)

In [11]:
from tokenizers import Tokenizer as HFTok

# Định nghĩa các hằng số hỗ trợ
MAX_LEN = tf_cfg.get('max_len', 74)
PAD_IDX = tf_cfg.get('pad_idx', 0)
BOS_IDX = 2
EOS_IDX = 3

tok = HFTok.from_file(TOKENIZER_PATH)

def translate_transformer(src_text: str) -> str:
    ids  = tok.encode(src_text).ids[:MAX_LEN]
    ids += [PAD_IDX] * (MAX_LEN - len(ids))
    src_t = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        out = transformer.translate_greedy(src_t, BOS_IDX, EOS_IDX, MAX_LEN)
    return tok.decode(out, skip_special_tokens=True)


def translate_lstm(src_text: str) -> str:
    if lstm_model is None:
        return '[LSTM model khong co san]'
    ids  = tok.encode(src_text).ids[:MAX_LEN]
    ids += [PAD_IDX] * (MAX_LEN - len(ids))
    src_t = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        # Encoder
        enc_h, enc_c = lstm_model.encoder(src_t)
        hidden = (enc_h, enc_c)

        # Greedy decode: truy cap truc tiep cac tang ben trong Decoder
        dec_input = torch.tensor([[BOS_IDX]], dtype=torch.long).to(DEVICE)
        out_ids   = []
        dec = lstm_model.decoder   # tham chieu den Decoder module

        for _ in range(MAX_LEN):
            embedded        = dec.embedding(dec_input)          # [1, 1, E]
            output, hidden  = dec.lstm(embedded, hidden)        # [1, 1, H], cap nhat hidden
            logit           = dec.output_projection(output)     # [1, 1, V]
            next_id         = logit[0, -1].argmax().item()
            if next_id == EOS_IDX:
                break
            out_ids.append(next_id)
            dec_input = torch.tensor([[next_id]], dtype=torch.long).to(DEVICE)

    return tok.decode(out_ids, skip_special_tokens=True)



# Thu nhanh
samples = [
    '<2vi> Hello , how are you ?',
    '<2ja> I love machine learning .',
    '<2zh> Thank you very much .',
]
print('Thu dich nhanh:\n')
for s in samples:
    print(f'  SRC  : {s}')
    print(f'  TF   : {translate_transformer(s)}')
    print(f'  LSTM : {translate_lstm(s)}')
    print()

Thu dich nhanh:

  SRC  : <2vi> Hello , how are you ?
  TF   : Xin chào , bạn sao ?
  LSTM : Xin chào , thế nào ?

  SRC  : <2ja> I love machine learning .
  TF   : 私は マシ ュー の 学 を 学 んだ 。
  LSTM : 私は 、 その 仕事 の 学 を 学 んだ 。

  SRC  : <2zh> Thank you very much .
  TF   : 谢谢 非常 感谢
  LSTM : 谢谢 你的



## Cell 5 – Tinh BLEU Score tren test.txt

In [12]:
!pip install sacrebleu tqdm

import sacrebleu
from tqdm.auto import tqdm
import os

# Tìm đường dẫn đúng của file test
possible_paths = [
    '/content/Multilingual_MT/data/processed/test.txt',
    '/content/Multilingual_MT/Multilingual_MT/data/processed/test.txt',
    './data/processed/test.txt'
]

FINAL_TEST_FILE = None
for p in possible_paths:
    if os.path.exists(p):
        FINAL_TEST_FILE = p
        break

if FINAL_TEST_FILE is None:
    print("❌ Không tìm thấy file test.txt!")
    print("Vui lòng chạy lệnh sau để tạo dữ liệu: !python src/prepare_data.py")
else:
    print(f"✅ Đã tìm thấy file test tại: {FINAL_TEST_FILE}")

def compute_bleu(translate_fn, test_file, label='Model'):
    refs, hyps = [], []
    with open(test_file, encoding='utf-8') as f:
        lines = [l.strip() for l in f if '\t' in l.strip()]

    for line in tqdm(lines, desc=f'  [{label}] Dang dich'):
        src, tgt = line.split('\t', 1)
        hyps.append(translate_fn(src))
        refs.append(tgt)

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score, hyps, refs

if FINAL_TEST_FILE:
    print('Tinh BLEU cho Transformer...')
    tf_bleu, tf_hyps, tf_refs = compute_bleu(translate_transformer, FINAL_TEST_FILE, 'Transformer')
    print(f'  Transformer BLEU: {tf_bleu:.2f}\n')

    if lstm_model is not None:
        print('Tinh BLEU cho LSTM Baseline...')
        lstm_bleu, lstm_hyps, _ = compute_bleu(translate_lstm, FINAL_TEST_FILE, 'LSTM')
        print(f'  LSTM Baseline BLEU: {lstm_bleu:.2f}')
    else:
        lstm_bleu = None
        print('Bo qua BLEU cho LSTM (khong co file .pt)')

✅ Đã tìm thấy file test tại: /content/Multilingual_MT/data/processed/test.txt
Tinh BLEU cho Transformer...


  [Transformer] Dang dich:   0%|          | 0/30000 [00:00<?, ?it/s]

  Transformer BLEU: 14.14

Tinh BLEU cho LSTM Baseline...


  [LSTM] Dang dich:   0%|          | 0/30000 [00:00<?, ?it/s]

  LSTM Baseline BLEU: 1.36


## Cell 6 – Bang so sanh chinh thuc

In [13]:
import pandas as pd

lstm_bleu_str = f'{lstm_bleu:.2f}' if lstm_bleu is not None else 'N/A'
tf_better     = tf_val_loss < lstm_val_loss

rows = [
    ['Val Loss (thap hon = tot hon)', f'{lstm_val_loss:.4f}', f'{tf_val_loss:.4f}',
     'Transformer' if tf_better else 'LSTM'],
    ['Perplexity (thap hon = tot hon)', f'{lstm_ppl:.2f}', f'{tf_ppl:.2f}',
     'Transformer' if tf_ppl < lstm_ppl else 'LSTM'],
    ['BLEU Score (cao hon = tot hon)', lstm_bleu_str, f'{tf_bleu:.2f}',
     'Transformer' if lstm_bleu is None or tf_bleu > lstm_bleu else 'LSTM'],
]

df = pd.DataFrame(rows, columns=['Metric', 'LSTM Baseline', 'Transformer', 'Model tot hon'])
print('\n' + '='*65)
print('  BANG SO SANH HIEU NANG 2 MO HINH')
print('='*65)
print(df.to_string(index=False))
print('='*65)
print(f'\n  Ket luan: Transformer cai thien Val Loss -{lstm_val_loss - tf_val_loss:.4f}')
print(f'            va Perplexity -{lstm_ppl - tf_ppl:.2f} so voi LSTM Baseline.')


  BANG SO SANH HIEU NANG 2 MO HINH
                         Metric LSTM Baseline Transformer Model tot hon
  Val Loss (thap hon = tot hon)        4.2456      4.1345   Transformer
Perplexity (thap hon = tot hon)         69.80       62.46   Transformer
 BLEU Score (cao hon = tot hon)          1.36       14.14   Transformer

  Ket luan: Transformer cai thien Val Loss -0.1112
            va Perplexity -7.34 so voi LSTM Baseline.


## Cell 7 – Phan tich loi (Error Analysis)

In [14]:
import sacrebleu

sentence_scores = [
    (sacrebleu.sentence_bleu(hyp, [ref]).score, hyp, ref)
    for hyp, ref in zip(tf_hyps, tf_refs)
]

worst = sorted(sentence_scores, key=lambda x: x[0])[:5]
best  = sorted(sentence_scores, key=lambda x: x[0], reverse=True)[:5]

print('5 CAU DICH TOT NHAT (Transformer)\n')
for i, (score, hyp, ref) in enumerate(best, 1):
    print(f'  [{i}] BLEU={score:.1f}')
    print(f'       REF : {ref}')
    print(f'       HYP : {hyp}')
    print()

print('\n5 CAU DICH TE NHAT (Transformer)\n')
for i, (score, hyp, ref) in enumerate(worst, 1):
    print(f'  [{i}] BLEU={score:.1f}')
    print(f'       REF : {ref}')
    print(f'       HYP : {hyp}')
    print()


5 CAU DICH TOT NHAT (Transformer)

  [1] BLEU=100.0
       REF : でも...
       HYP : でも ...

  [2] BLEU=100.0
       REF : Agenda item 55 (a)
       HYP : Agenda item 55 ( a )

  [3] BLEU=100.0
       REF : This is from a global survey .
       HYP : This is from a global survey .

  [4] BLEU=100.0
       REF : What do you want?
       HYP : What do you want ?

  [5] BLEU=100.0
       REF : I felt safe.
       HYP : I felt safe .


5 CAU DICH TE NHAT (Transformer)

  [1] BLEU=0.0
       REF : ここで休憩を パーティーはこれからだよ
       HYP : ちょっと した くない けど 、 もう 戻 らない と

  [2] BLEU=0.0
       REF : 92nd plenary meeting 1 July
       HYP : A / 58 / CRP . 4 .

  [3] BLEU=0.0
       REF : 我们被清除 一百万一个星期。
       HYP : 他 用 了

  [4] BLEU=0.0
       REF : 馬、食事と必要な物の調達だ
       HYP : いい 馬 は いい 馬 だ 食 事

  [5] BLEU=0.0
       REF : -已布下天罗地网
       HYP : - 我们 都 被 开 了

